In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'torch', 'torchvision']
imports = {'pillow': 'PIL'}
pinned = {}
fallbacks = {'torch': 'torch==2.11.0', 'torchvision': 'torchvision==0.26.0'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'pytorch' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'torch.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Multilayer Perceptrons

The softmax regression of that section classifies a
Fashion-MNIST image by adding up evidence from its pixels, with one weight
for each pixel and class. With 7,850 parameters it reaches about 83%
validation accuracy. Its limit lies in how it adds the evidence: each input
moves a class score by the same amount whatever the other inputs are. Yet in
an image, whether a bright pixel belongs to a sleeve, a strap, or the
background depends on the pixels around it. Can a model learn to combine its
inputs in this way, and what does it need beyond a weighted sum?

The smallest problem of this kind has four points. Label each corner of the
unit square by whether its two coordinates differ, the *exclusive or* (XOR)
of two bits. No line separates the two classes, so every linear classifier
gets at least one of the four points wrong.

A *multilayer perceptron* (MLP) removes this limit by computing new features
before it classifies. A hidden layer applies an affine map followed by a
fixed nonlinear function, and an output layer, which is softmax regression,
classifies the result. Both layers are trained together, with the loss and
the minibatch stochastic gradient descent (SGD) of the previous chapter.
This section defines the model and builds a network with two hidden units
that computes XOR. It then compares the nonlinear functions in common use
and asks which functions such networks can represent. The next section
trains an MLP on Fashion-MNIST.

In [ ]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

## Limits of Linear Models

### Monotonicity and Interactions

To see what a linear model cannot express, we look again at how it scores a
class. In softmax regression (that section), an input
$\mathbf{x} \in \mathbb{R}^d$ gives class $j$ the score

$$o_j = \sum_{i=1}^d x_i w_{ij} + b_j.$$

Each score is therefore *monotonic* in each input: it rises with $x_i$ if
$w_{ij} > 0$ and falls if $w_{ij} < 0$. Moreover, the inputs do not
interact: raising $x_i$ by one changes $o_j$ by $w_{ij}$, whatever the other
inputs are. Both properties often fail. The risk of illness rises as body
temperature departs from 37°C in either direction. This case we can repair
by hand, using the distance from 37°C as the input. Images resist such
repairs. Whether a bright pixel is evidence for a shirt or for a bag depends
on its neighbors, and writing down features that capture all such
dependencies is impractical. We want the model to learn its features from
data instead.

### The XOR Problem

The smallest problem in which inputs interact has two binary inputs and the
label

| $x_1$ | $x_2$ | $y$ |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

which is 1 if exactly one input is 1. Changing $x_1$ from 0 to 1 changes the
label from 0 to 1 when $x_2 = 0$, but from 1 to 0 when $x_2 = 1$. With two
classes, softmax regression predicts class 1 where the difference of the two
logits is positive (that section). This difference is an affine
score $o = w_1 x_1 + w_2 x_2 + b$, so the decision boundary is the line
$o = 0$.

**Proposition.** For all $w_1, w_2, b$, the rule "predict class 1 if
$w_1 x_1 + w_2 x_2 + b > 0$" misclassifies at least one of the four points
of XOR.

**Proof.** Suppose that some $w_1, w_2, b$ classify all four points
correctly. The two points of class 1 give $w_2 + b > 0$ and $w_1 + b > 0$,
and adding the two inequalities gives $w_1 + w_2 + 2b > 0$. The two points
of class 0 give $b \leq 0$ and $w_1 + w_2 + b \leq 0$, and adding these gives
$w_1 + w_2 + 2b \leq 0$, a contradiction. $\blacksquare$

The proof has a picture. Both pairs of points have the midpoint
$(1/2, 1/2)$, where an affine score equals the average of its values at
either pair, so it would have to be positive and not positive at once. A
line can classify three of the four points correctly: the line
$x_1 + x_2 = 3/2$, for example, misclassifies only $(0, 0)$.

## Hidden Layers

### The Model

An MLP first computes $h$ new features, each an affine function of the input
followed by a fixed nonlinear function $\phi$, and then applies a linear
model to them. The features form a *hidden layer* of *hidden units*, so
called because their values are neither given by the data nor reported as
outputs. the figure shows an MLP with four inputs, five hidden units,
and three outputs. Every input feeds every hidden unit, and every hidden
unit feeds every output, so both layers are fully connected. The inputs
involve no computation, so we count only the layers with parameters: this
MLP has two layers, one of them hidden. The name comes from the perceptron,
an early linear classifier, of which an MLP stacks several layers.

![An MLP with four inputs, a hidden layer of five units, and three outputs. Both layers are fully connected.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mlp.svg)

We keep the layout of that section: the $n$
examples of a minibatch are the rows of $\mathbf{X} \in \mathbb{R}^{n \times d}$,
and each weight matrix has one row per input and one column per output of
its layer. With a superscript numbering the layers, the hidden layer has
weights $\mathbf{W}^{(1)} \in \mathbb{R}^{d \times h}$ and biases
$\mathbf{b}^{(1)} \in \mathbb{R}^{h}$, the output layer has weights
$\mathbf{W}^{(2)} \in \mathbb{R}^{h \times q}$ and biases
$\mathbf{b}^{(2)} \in \mathbb{R}^{q}$, and the MLP computes

$$
\begin{aligned}
\mathbf{H} &= \phi\left(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}\right),\\
\mathbf{O} &= \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}.
\end{aligned}
$$

The biases are added to every row by broadcasting, and $\phi$ acts on each
entry separately. Row $i$ of $\mathbf{H} \in \mathbb{R}^{n \times h}$ is the
*hidden representation* of example $i$, and row $i$ of
$\mathbf{O} \in \mathbb{R}^{n \times q}$ holds its logits, which the softmax
turns into probabilities as before. For a single example $\mathbf{x}$, the
model reads $\mathbf{h} = \phi(\mathbf{W}^{(1)\top}\mathbf{x} + \mathbf{b}^{(1)})$
and $\mathbf{o} = \mathbf{W}^{(2)\top}\mathbf{h} + \mathbf{b}^{(2)}$.

The function $\phi$ is the *activation function*. Its inputs, the entries of
$\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}$, are called
*pre-activations*, and its outputs *activations*. Minibatch SGD fits all four parameter arrays
together on the cross-entropy loss, with gradients from automatic
differentiation, so the network learns a representation and a linear
classifier of that representation at once.

The hidden layer adds parameters. With $d$ inputs, $h$ hidden units, and
$q$ outputs, an MLP has $(d + 1)h + (h + 1)q$ parameters, against
$(d + 1)q$ for softmax regression. For Fashion-MNIST, with $d = 784$ and
$q = 10$, a hidden layer of $h = 256$ units gives
$785 \cdot 256 + 257 \cdot 10 = 203{,}530$ parameters, 26 times the 7,850 of
softmax regression.

### The Role of the Nonlinearity

Why apply $\phi$ at all? Suppose that we omit it, so that
$\mathbf{H} = \mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}$. Substituting
$\mathbf{H}$ into the output layer gives

$$\mathbf{O} = \left(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}\right)\mathbf{W}^{(2)} + \mathbf{b}^{(2)} = \mathbf{X}\mathbf{W} + \mathbf{b},$$

where $\mathbf{W} = \mathbf{W}^{(1)}\mathbf{W}^{(2)}$ and
$\mathbf{b} = \mathbf{W}^{(2)\top}\mathbf{b}^{(1)} + \mathbf{b}^{(2)}$. The
two layers compute an affine function of the input, one that softmax
regression computes with the weights $\mathbf{W}$ and the bias $\mathbf{b}$.
The hidden layer adds parameters but no new functions (it can even remove
some, Exercise 1), and the network still fails
on XOR.

With a nonlinear $\phi$, the substitution no longer simplifies. The simplest
useful choice, and the default in this book, is the *rectified linear unit*
(ReLU),

$$\mathrm{ReLU}(z) = \max(z, 0),$$

which keeps positive values and replaces negative ones by zero.

### Solving XOR

Two hidden ReLU units suffice for XOR. One choice of weights
[@Goodfellow.Bengio.Courville.2016] is

$$\mathbf{W}^{(1)} = \begin{pmatrix} 1 & 1 \\ 1 & 1 \end{pmatrix}, \quad \mathbf{b}^{(1)} = (0, -1), \quad \mathbf{w}^{(2)} = (1, -2), \quad b^{(2)} = 0,$$

where the output layer has a single output $o$, so its weights form one
column $\mathbf{w}^{(2)}$. Both hidden units see the sum $s = x_1 + x_2$.
The first, $h_1 = \mathrm{ReLU}(s) = s$, counts the inputs that are 1. The
second, $h_2 = \mathrm{ReLU}(s - 1)$, is positive only when both inputs are
1, and the output $o = h_1 - 2h_2$ subtracts 2 in that case. We check the
network on the four inputs, stored as the rows of $\mathbf{X}$.

In [2]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
W1, b1 = np.array([[1, 1], [1, 1]]), np.array([0, -1])
w2, b2 = np.array([1, -2]), 0
H = np.maximum(X @ W1 + b1, 0)  # hidden layer with ReLU
o = H @ w2 + b2  # output layer
for x, h, out in zip(X, H, o):
    print(f'x = {x}, h = {h}, o = {out}')

x = [0 0], h = [0 0], o = 0
x = [0 1], h = [1 0], o = 1
x = [1 0], h = [1 0], o = 1
x = [1 1], h = [2 1], o = 0


The output equals the XOR label on every input. As a function of $s$, the
output $o = \mathrm{ReLU}(s) - 2\,\mathrm{ReLU}(s - 1)$ rises from 0 at
$s = 0$ to 1 at $s = 1$ and falls back to 0 at $s = 2$. The second unit
*folds* the line of values of $s$ at $s = 1$, so that its two ends reach the
same output. In the hidden space (the figure, right), the two
inputs of class 1 land on the same point $(1, 0)$, and the line
$h_1 - 2h_2 = 1/2$ separates the classes. In the input space (left), the
network predicts class 1 in the band $1/2 < x_1 + x_2 < 3/2$, a region that
no single line can bound.

![XOR and the two-unit ReLU network that computes it. Left: the four inputs, labeled by XOR (blue: class 0, orange: class 1); the network predicts class 1 in the shaded band between the dashed lines $x_1 + x_2 = 1/2$ and $x_1 + x_2 = 3/2$. Right: the same inputs after the hidden layer $\mathbf{h} = \mathrm{ReLU}(\mathbf{W}^{(1)\top}\mathbf{x} + \mathbf{b}^{(1)})$, each marked with the input it came from; the two inputs of class 1 land on the same point $(1, 0)$, and the dashed line $h_1 - 2h_2 = 1/2$ separates the classes.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-xor.svg)

The hidden layer has re-represented the inputs so that a linear classifier
suffices. Here we chose its weights by hand. For a problem such as
Fashion-MNIST nobody could, and training chooses them instead, adjusting the
hidden layer together with the output layer.

### Deeper Networks

Nothing stops us from stacking more hidden layers. Starting from
$\mathbf{H}^{(0)} = \mathbf{X}$, hidden layer $k$ computes

$$\mathbf{H}^{(k)} = \phi\left(\mathbf{H}^{(k-1)}\mathbf{W}^{(k)} + \mathbf{b}^{(k)}\right), \quad k = 1, \ldots, K-1,$$

and the output layer computes
$\mathbf{O} = \mathbf{H}^{(K-1)}\mathbf{W}^{(K)} + \mathbf{b}^{(K)}$. Such a
network has $K$ layers, $K - 1$ of them hidden, and each hidden layer may
have its own width. Each hidden layer computes a new representation from the
one below it, and the output layer classifies the last one. For $K = 2$ this
is the equation. Whether several narrow layers do better than one wide
layer is a question we return to at the end of this section.

## Activation Functions

Any nonlinear $\phi$ prevents the collapse into one affine map, but the
choice matters for training. Gradient descent changes each parameter in
proportion to the derivative of the loss $l$ with respect to it. For one
example, hidden unit $j$ computes $h_j = \phi(z_j)$ from its pre-activation
$z_j = \sum_i x_i w^{(1)}_{ij} + b^{(1)}_j$, where $w^{(1)}_{ij}$ is an
entry of $\mathbf{W}^{(1)}$. By the chain rule,

$$\frac{\partial l}{\partial w^{(1)}_{ij}} = \frac{\partial l}{\partial h_j}\,\phi'(z_j)\,x_i.$$

Every weight into unit $j$ is scaled by the same factor $\phi'(z_j)$. Where
it is close to zero, none of these weights moves, whatever the loss.
the figure shows four activation functions in common
use, with their derivatives.

![Four activation functions $\phi$ (left) and their derivatives $\phi'$ (right). Top: the ReLU and its smooth relative, the GELU. Bottom: the sigmoid and tanh, which squash their inputs into bounded intervals. The two derivative panels share one vertical scale: the derivative of the sigmoid never exceeds $1/4$, and the derivatives of the sigmoid and tanh vanish in both tails.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-activations.svg)

### ReLU and GELU

The ReLU passes positive pre-activations unchanged and sets negative ones to
zero. Its derivative is 1 for $z > 0$ and 0 for $z < 0$. At $z = 0$, where
the ReLU has no derivative, the built-in ReLU uses 0, a choice that rarely
matters because a pre-activation is seldom exactly zero. For every positive
pre-activation, however large, the gradient passes through a ReLU unit
unchanged. This is the main reason why the ReLU replaced the sigmoid,
discussed below, as the default activation of hidden layers
[@Nair.Hinton.2010; @Glorot.Bordes.Bengio.2011]. It is also cheap: a
single comparison.

The flat side has a cost. A unit whose pre-activation is negative for every
training example outputs zero, receives zero gradient, and never changes
again. Such a unit is called *dead*. The leaky ReLU, $\max(z, \alpha z)$
with a small slope such as $\alpha = 0.01$, keeps a small gradient for
negative inputs, so its units cannot die.

The *Gaussian error linear unit* (GELU) is a smooth relative of the ReLU,

$$\mathrm{GELU}(z) = z\,\Phi(z),$$

where $\Phi$ is the cumulative distribution function of the standard normal
distribution. Since $\Phi(z)$ rises smoothly from 0 for very negative $z$ to
1 for very positive $z$, the GELU matches the ReLU for $|z|$ beyond about 3
and rounds off its corner in between. Its slope at $z = 0$ is 1/2, and
unlike the ReLU it dips below zero, to about $-0.17$ near $z = -0.75$. Many
transformer models use the GELU [@Hendrycks.Gimpel.2016]. Some
libraries compute it by default with an approximation built from the tanh,
which differs from $z\,\Phi(z)$ by less than $5 \cdot 10^{-4}$. A close
relative, the *SiLU* $z\,\sigma(z)$ with the sigmoid $\sigma$ of the next
subsection, is used in a gated form, SwiGLU, by many recent language models
(that section).

### Sigmoid and Tanh

The sigmoid, $\sigma(z) = 1/(1 + \exp(-z))$, turns the logit of a two-class
classifier into a probability (that section): it maps the real line
onto the interval $(0, 1)$. Early neural networks used it in hidden layers
as a smooth version of a threshold unit, which outputs 1 when its input
exceeds a threshold and 0 otherwise [@McCulloch.Pitts.1943]. Unlike the
threshold, the sigmoid has a derivative that gradient descent can use,

$$\sigma'(z) = \sigma(z)\,\bigl(1 - \sigma(z)\bigr).$$

It is largest at $z = 0$, where it equals $1/4$, and approaches zero in both
tails. A unit whose pre-activation is large in magnitude is *saturated*: its
output barely changes with its input, so it passes almost no gradient to the
weights below it.

Even at $z = 0$, every sigmoid unit between a weight and the loss shrinks
the gradient by a factor of 4. Through a chain of ten sigmoid units with
weights of 1, the gradient therefore shrinks by a factor of at least
$4^{10} \approx 10^6$. Such *vanishing gradients* made deep sigmoid networks
hard to train; that section analyzes them and their
remedies.

The *hyperbolic tangent*,

$$\tanh(z) = \frac{1 - \exp(-2z)}{1 + \exp(-2z)},$$

maps the real line onto $(-1, 1)$. It is a sigmoid stretched to this
interval and centered at zero (Exercise 3), and
its derivative, $1 - \tanh^2(z)$, reaches 1 at $z = 0$ and vanishes in both
tails as well. Today the sigmoid serves mainly to produce probabilities and
gates, numbers between 0 and 1 that decide what fraction of a signal passes,
and tanh to keep the state of recurrent networks bounded.

We check the four derivatives with automatic differentiation
(that section) at $z = -4, -1, 0, 1, 4$. Each activation function
acts on every entry separately, so the gradient of the sum of its outputs
holds its derivative at each point.

In [3]:
z = torch.tensor([-4.0, -1.0, 0.0, 1.0, 4.0], requires_grad=True)
activations = {'ReLU': torch.relu, 'sigmoid': torch.sigmoid,
               'tanh': torch.tanh, 'GELU': F.gelu}
for name, phi in activations.items():
    # returns the gradient rather than accumulating it in z.grad
    grad, = torch.autograd.grad(phi(z).sum(), z)
    print(f'{name:8s}', ' '.join(f'{g:7.4f}' for g in grad.tolist()))

ReLU      0.0000  0.0000  0.0000  1.0000  1.0000
sigmoid   0.0177  0.1966  0.2500  0.1966  0.0177
tanh      0.0013  0.4200  1.0000  0.4200  0.0013
GELU     -0.0005 -0.0833  0.5000  1.0833  1.0005


The ReLU passes a gradient of 0 or 1, with 0 at $z = 0$. The sigmoid passes
at most 1/4 of the gradient and tanh at most all of it, but at $z = \pm 4$
both are saturated, passing 0.018 and 0.0013 of it. The GELU passes a small
negative gradient at $z = -1$.

## Expressive Power

### Universal Approximation

Two hidden units sufficed for XOR. Can a network with one hidden layer
approximate every function that we might want to learn? For a single input
$x$ and ReLU units, a picture answers the question. A hidden unit with input
weight 1, bias $-t$, and output weight $a$ adds $a\,\mathrm{ReLU}(x - t)$ to
the output. This term is a *hinge*: zero to the left of its joint $t$ and a
line of slope $a$ to the right of it. A sum of $h$ hinges and a constant is
continuous and linear between the joints, so it has at most $h + 1$ linear
pieces (the figure). Conversely, every polyline on a
closed interval is such a sum: one hinge at the left end with the slope of
the first piece, and one hinge at each later joint with the change of slope
there.

![Approximation by hinges. Left: three hidden units, each contributing a hinge $a_j\,\mathrm{ReLU}(x - t_j)$ whose joint $t_j$ is marked. Right: a line plus the three hinges is a polyline with 4 pieces (blue) that follows a smooth target (gray); the shaded area is the error, which shrinks as more joints are added.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-uat-hinges.svg)

The approximation result for one input follows. A continuous function $f$ on
a closed interval changes by less than any given $\epsilon > 0$ between
points that are close enough together. The polyline through the values of
$f$ at such closely spaced points therefore stays within $\epsilon$ of $f$
on the whole interval, and a hidden layer with one unit per joint computes
it. The same conclusion holds for any number of inputs, on any closed and
bounded set, and for every continuous activation function that is not a
polynomial (@Cybenko.1989 for sigmoids,
@Leshno.Lin.Pinkus.ea.1993 in general). Polynomials must be
excluded, because with a polynomial activation function the network computes
a polynomial of bounded degree, and such polynomials cannot approximate
every continuous function. This result is the *universal approximation
theorem*. Exercise 6 proves it for two
inputs.

### Limits of the Theorem

The theorem says that good weights exist, and it leaves three questions
open. The first is width. The construction above uses one unit per joint,
and a target that changes direction often needs many joints. With many
inputs, constructions of this kind cover the input space with small cells,
and the number of cells, and with it the number of hidden units, can grow
exponentially with the number of inputs. The second question is training:
the theorem does not say that minibatch SGD, started from random weights,
finds good weights. The third is generalization. Approximating a function
that we know everywhere differs from learning it from $n$ examples, and a
network flexible enough to approximate any function can also fit the noise
in its training labels.

### Depth

Width adds linear pieces one unit at a time: on one input, $h$ hidden units
give at most $h + 1$ pieces. Depth can multiply them. Consider the tent

$$g(z) = 2\,\mathrm{ReLU}(z) - 4\,\mathrm{ReLU}(z - 1/2),$$

which rises from 0 to 1 on $[0, 1/2]$ and falls back to 0 on $[1/2, 1]$.
It is the XOR network read along the diagonal: with $s = 2z$, the output
$\mathrm{ReLU}(s) - 2\,\mathrm{ReLU}(s - 1)$ equals $g(z)$. Now feed $g$ its
own output. As $z$ runs from 0 to 1/2, $g(z)$ runs from 0 to 1, so
$g(g(z))$ traces the whole tent. As $z$ runs on from 1/2 to 1, $g(z)$ runs
back from 1 to 0, and $g(g(z))$ traces the tent again in reverse. The first
layer folds the interval $[0, 1]$ in half, and whatever comes after it is
computed on both halves. Each composition with $g$ doubles the number of
pieces (the figure), and it takes one more hidden
layer of two units, since the output map of one layer and the affine map of
the next merge into a single affine map.

![Depth folds the input. The tent $g$, computed by one hidden layer of two ReLU units, and its compositions $g \circ g$ and $g \circ g \circ g$, computed by two and three such layers. Each composition splits every piece of the previous function (dashed) in two.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-depth-folding.svg)

Thus composing $g$ with itself $m$ times gives a zigzag with $2^m$ pieces,
computed by $m$ hidden layers of two units, $2m$ hidden units in all. A
single hidden layer needs at least $2^m - 1$ units for the same function,
since $h$ units give at most $h + 1$ pieces. For $m = 10$, this means 20
units in ten hidden layers against at least 1,023 in one.
@Telgarsky.2016 used this construction to prove that some functions
computed by deep, narrow networks require exponentially wide shallow
networks. Depth does not help with every function, and nothing guarantees
that training finds such compact solutions.

We can now build models that represent nonlinear functions: hidden layers of
affine maps and activation functions, followed by the linear classifier of
softmax regression. Whether training finds good weights, and whether they
predict well on new data, are questions for experiment. The next section
trains an MLP on Fashion-MNIST and compares it with softmax regression.

## Summary

A linear classifier scores each class with a weighted sum of the inputs, so
it cannot represent interactions between inputs, and the four points of XOR
already defeat it. A multilayer perceptron first computes a hidden
representation $\mathbf{H} = \phi(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)})$
and then classifies it linearly,
$\mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}$, training both
layers together. Without the activation function $\phi$, the layers
collapse into one affine map; with it, two ReLU units fold the XOR inputs so
that a line separates them. The ReLU passes gradients of 0 or 1 and is the
default for hidden layers, while the sigmoid and tanh saturate and pass at
most 1/4 and 1 of the gradient. One hidden layer can approximate any
continuous function, possibly only with very many units, and depth can
produce exponentially many linear pieces from a linear number of units.

## Exercises

1. **Linear networks.** Without activation functions, stacking layers
   composes affine maps into another affine map. Consider such a network
   with input dimension $d$, output dimension $q$, and hidden widths
   $h_1, \ldots, h_{K-1}$. Show that it represents exactly the affine maps
   $\mathbf{X}\mathbf{W} + \mathbf{b}$ whose weight matrix $\mathbf{W}$ has
   rank at most $\min(h_1, \ldots, h_{K-1})$. Use this result to give an
   example in which adding a hidden layer to a linear model strictly shrinks
   its function class.
1. **XOR by hand.** the figure shows a two-unit ReLU network
   that computes XOR by mapping the two inputs of class 1 to the same hidden
   point.
    1. Find weights for a two-unit ReLU network that also computes XOR but
       maps the two inputs of class 1 to *distinct* hidden points, and verify
       it on the four inputs.
    1. Show that no network with a single ReLU hidden unit and a linear
       output classifies XOR correctly, whatever threshold it applies to its
       output.
1. **Sigmoid and tanh.** The two squashing functions are closely related.
    1. Show that $\tanh(z) = 2\sigma(2z) - 1$.
    1. Show that for every MLP with tanh hidden activations there is an MLP
       of the same architecture with sigmoid hidden activations that
       computes the same function, and conversely. State how the weights and
       biases change.
1. [code] **Vanishing gradients.** Consider a chain of $K$ sigmoid units of
   width one, $h_k = \sigma(z_k)$ with pre-activation
   $z_k = w_k h_{k-1} + b_k$. By the chain rule,

    $$\frac{\partial h_K}{\partial h_0} = \prod_{k=1}^{K} w_k \sigma'(z_k).$$

    1. Explain why the weight feeding a *saturated* unit, one operating at
       large $|z_k|$, receives almost no gradient whatever the loss, and why
       a single saturated unit also stalls every earlier layer of the chain.
    1. Choose the biases so that every pre-activation equals $0$, where
       $\sigma'$ attains its maximum $1/4$. Compute the product for
       $K = 1, \ldots, 20$, once with every $w_k = 1$ and once with every
       $w_k = 4$, and plot both curves on a logarithmic vertical axis.
    1. Repeat with every pre-activation equal to $5$ and every $w_k = 1$.
       Find the weight $w$ for which $w \sigma'(5) = 1$, and compare it with
       the weight that keeps the product constant at the origin. State what
       the comparison implies for training deep sigmoid networks in either
       regime.
1. **Depth and linear pieces.** ● A one-hidden-layer ReLU network with $h$
   hidden units on the real line has at most $h + 1$ linear pieces, while
   each composition with the two-unit tent of
   the figure doubles the number of pieces. This
   problem makes the gain from depth exact for a network with three units
   per layer.
    1. Take three hidden units $\mathrm{ReLU}(w_j x + b_j)$ with input
       weights $w_1 = w_2 = 1$ and $w_3 = -1$ and with joints $-b_j/w_j$ at
       $1/6$, $2/6$, and $4/6$. Choose output weights and an output bias so
       that the network's output $f$ takes the values $0, 1, 0, 1, 0$ at
       $x = 0, 1/6, 2/6, 4/6, 1$.
    1. Count the linear pieces of $f \circ f$ on $[0, 1]$, and of the
       $m$-fold composition $f \circ \cdots \circ f$. Explain, in terms of
       how composing with $f$ folds the graph, why each composition
       multiplies the count rather than adding to it.
    1. The $m$-fold composition is computed by a network with $m$ hidden
       layers of three units each. How many hidden units would a network
       with a single hidden layer need to represent the same function?
       Compare the parameter counts of the two networks.

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 4.8.*
1. **One hidden layer suffices.** ● On the real line, a single hidden layer
   approximates a continuous function by a polyline, one hinge per joint.
   For two inputs, a natural extension uses two hidden layers: the first
   builds steps along the coordinate axes from pairs of ReLU units, and the
   second combines them into localized bumps, which are then summed. Show that the second hidden layer is unnecessary.
    1. Let $\psi$ be continuous on $\mathbb{R}$ and let
       $\mathbf{w} \in \mathbb{R}^2$. Show that a single hidden layer
       approximates the *ridge function* $\psi(\mathbf{w}^\top \mathbf{x})$,
       which is constant along every line orthogonal to $\mathbf{w}$,
       uniformly on any compact subset of the plane.
    1. The Stone–Weierstrass theorem states the following. Let a set of
       continuous functions on a compact set be closed under sums, products,
       and scalar multiples, contain the constants, and contain, for any two
       distinct points, a function that takes different values at them. Then
       every continuous function on the compact set is a uniform limit of
       functions in the set. Show that the finite linear combinations of
       $\cos(\boldsymbol{\omega}^\top \mathbf{x})$ and
       $\sin(\boldsymbol{\omega}^\top \mathbf{x})$, with
       $\boldsymbol{\omega} \in \mathbb{R}^2$, form such a set.
    1. Conclude that a single hidden layer approximates any continuous
       function on a compact subset of the plane.

    *Adapted from Michael Nielsen,
    [Neural Networks and Deep Learning](http://neuralnetworksanddeeplearning.com/chap4.html),
    Chapter 4.*
1. [code] **Weight scale and saturation.** ● Take the chain of Exercise
   4 with $K = 20$, every weight
   $w_k = w$, and every bias $b_k = -w/2$, so that the input $h_0 = 1/2$
   gives $z_k = 0$ at every layer.
    1. Express $\partial h_K / \partial h_0$ as a function of $w$ for the
       input $h_0 = 1/2$. For which $w$ does it not vanish as $K$ grows?
    1. Implement the chain and run it from $h_0 = 0.6$ with $w = 4$ and with
       $w = 8$. Report $h_{20}$ and $\partial h_{20} / \partial h_0$ in each
       case.
    1. Treat the map $h_{k-1} \mapsto h_k$ as a one-dimensional iteration
       with fixed point $1/2$, and use its slope there to explain the
       difference between the two runs. State what the result implies for
       countering vanishing gradients in a sigmoid network by scaling up its
       weights.

[Discussions](https://d2l.discourse.group/t/91)